# CTKM Extractor — chạy bản **C++** trên Google Colab

Colab không phải môi trường chỉ dành cho Python: nó là một máy ảo Ubuntu có `sudo`,
nên `apt`, `cmake`, `g++` chạy bình thường.

Notebook gồm hai mức, chạy tuần tự từ trên xuống:

| Mức | Nội dung | Cần model? | Thời gian |
| --- | --- | --- | --- |
| **1** | Build + chạy bằng engine fallback `tesseract` | Không | ~3 phút |
| **2** | Export `vietocr.onnx` + `vocab`, lấy `det.onnx`, bật engine mặc định `paddle_vietocr` | Có | ~10 phút |

Thời gian đo trên 2 CPU core (đúng cấu hình Colab free): `cmake` configure ~3 giây,
build đầy đủ kèm tải Catch2 **~70 giây**.

> **Phiên Colab là tạm**. Ngắt kết nối là mất sạch `/content`. Cell cuối cùng lưu
> model sang Google Drive để lần sau khỏi export lại — cả `vietocr.onnx` lẫn
> `det.onnx` đều không đổi giữa các lần chạy.
> **Cần bản C++ có bản sửa chuẩn hoá ảnh của `OnnxVietOCR`** (chia 255 thay vì
> đưa về `[-1, 1]`, và làm tròn bề rộng lên bội số của 10). Thiếu bản sửa đó thì
> mọi thứ vẫn chạy nhưng VietOCR trả ra text rác. Bản sửa nằm cùng PR với
> notebook này; clone `main` sau khi PR được merge là có sẵn.


---
# Mức 1 — build và chạy, không cần model

## Cell 1 — lấy code

In [ ]:
!git clone -q https://github.com/Ngoc-LM/OCR_Extractor.git /content/OCR_Extractor
%cd /content/OCR_Extractor
!git log --oneline -3

## Cell 2 — thư viện hệ thống

`libopencv-dev` là gói nặng nhất (~1 phút). `tesseract-ocr-vie` bắt buộc: thiếu gói
tiếng Việt thì OCR mất sạch dấu.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y cmake g++ libopencv-dev libtesseract-dev libleptonica-dev \
                        tesseract-ocr tesseract-ocr-vie
!tesseract --list-langs

## Cell 3 — build + test

Chưa có ONNXRuntime nên CMake in cảnh báo *"Không tìm thấy ONNXRuntime — provider
'paddle_vietocr' sẽ không khả dụng"*. **Đúng như mong đợi ở mức 1** — chương trình
vẫn build và chạy được bằng Tesseract.

Catch2 được tải qua FetchContent nên cell này cần mạng ở lần chạy đầu.

In [ ]:
%cd /content/OCR_Extractor/cpp
!cmake -B build
!cmake --build build -j2
!ctest --test-dir build --output-on-failure | tail -5
!./build/ctkm_extractor --list-engines

## Cell 4 — đưa ảnh vào

Upload thẳng file PNG/JPG, hoặc upload PDF rồi render 300 DPI ở cell kế tiếp.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# Chỉ chạy cell này nếu bạn upload PDF. Bỏ qua nếu đã có sẵn PNG/JPG.
!pip install -q pymupdf
import fitz, glob
pdf = glob.glob("/content/OCR_Extractor/cpp/*.pdf") + glob.glob("/content/*.pdf")
fitz.open(pdf[0])[0].get_pixmap(dpi=300).save("/content/input.png")

from PIL import Image
print("Kích thước:", Image.open("/content/input.png").size)   # A4 @300dpi ~ (2480, 3508)

## Cell 5 — chạy bằng Tesseract

`--debug` in raw OCR text, bảng đã dựng và nguồn của từng field ra stderr — đây là
chỗ để soi khi kết quả sai.

Thêm `--no-binarize` nếu ảnh có **watermark**: adaptive threshold biến nét watermark
mờ thành nét đen đặc đè lên chữ. Đo trên biểu mẫu BM.12 thật, cờ này đưa bản C++ từ
8/11 lên **11/11 field đúng**. Lưu ý cờ này chỉ có lợi với Tesseract — với engine
mặc định ở mức 2 thì **giữ nguyên mặc định** (xem `cpp/README.md`).

In [ ]:
!./build/ctkm_extractor --image /content/input.png --out /content/result_tesseract.json \
                        --engine tesseract --no-binarize --debug

import json, pathlib

out = pathlib.Path("/content/result_tesseract.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


---
# Mức 2 — bật engine mặc định `paddle_vietocr`

Bản C++ **không** dùng `paddleocr`/`torch` lúc chạy. Nó cần 3 file trong `cpp/models/`:

| File | Nguồn |
| --- | --- |
| `det.onnx` | Model detect DB của PP-OCR, lấy sẵn từ wheel `rapidocr-onnxruntime` |
| `vietocr.onnx` | Tự export từ `vietocr` bằng chính notebook này |
| `vietocr_vocab.txt` | Xuất cùng lúc với `vietocr.onnx` |

## Cell 6 — ONNXRuntime prebuilt

In [ ]:
!curl -sL -o /tmp/ort.tgz https://github.com/microsoft/onnxruntime/releases/download/v1.20.1/onnxruntime-linux-x64-1.20.1.tgz
!tar xzf /tmp/ort.tgz -C /tmp
!rm -rf /opt/onnxruntime && mv /tmp/onnxruntime-linux-x64-1.20.1 /opt/onnxruntime
!ls /opt/onnxruntime/lib/libonnxruntime.so*

## Cell 7 — `det.onnx`

Lấy từ wheel `rapidocr-onnxruntime` (4.7 MB), **không cần cài PaddleOCR hay chạy
`paddle2onnx`**. Đây là model DB của PP-OCRv4 đã export sẵn.

In [ ]:
!pip download --no-deps -q -d /tmp/rap rapidocr-onnxruntime
!cd /tmp/rap && unzip -oq *.whl
!cp /tmp/rap/rapidocr_onnxruntime/models/ch_PP-OCRv4_det_infer.onnx \
    /content/OCR_Extractor/cpp/models/det.onnx
!ls -lh /content/OCR_Extractor/cpp/models/det.onnx

## Cell 8 — cài `vietocr` (chỉ để export, không cần lúc chạy C++)

**Bắt buộc dùng `--no-deps`.** `vietocr` 0.3.13 ghim cứng `pillow==10.2.0`, còn Colab
đã cài sẵn Pillow 11+. Cài bình thường sẽ **hạ cấp Pillow ngay dưới chân kernel đang
chạy**: `PIL._util` bản mới đã nạp sẵn trong bộ nhớ, còn `PIL/ImageFont.py` bản
10.2.0 vừa ghi xuống đĩa lại `import is_directory` — hàm đã bị gỡ từ Pillow 10. Kết
quả là ở cell sau:

```
ImportError: cannot import name 'is_directory' from 'PIL._util'
```

Chuỗi import của `Predictor` thật ra **chỉ cần** `torch`, `torchvision`, `PIL`,
`numpy`, `einops`, `gdown`, `requests`, `yaml`, `tqdm`. Colab có sẵn tất cả trừ
`einops` và `gdown` — hai gói này không đụng tới Pillow. Những gói còn lại mà
`vietocr` khai báo (`albumentations`, `imgaug`, `lmdb`, `scikit-image`,
`prefetch-generator`) chỉ dùng cho phần **huấn luyện**, không nằm trong đường
inference.

Nhờ vậy **không phải restart runtime**.

In [ ]:
# --no-deps: KHÔNG để vietocr hạ cấp Pillow xuống 10.2.0 (xem giải thích ở trên).
!pip install -q --no-deps vietocr
!pip install -q einops gdown onnx onnxruntime onnxscript

# Kiểm tra Pillow còn nguyên vẹn trước khi đi tiếp.
import PIL
from PIL import ImageFont          # dòng này chính là chỗ vỡ nếu Pillow bị hạ cấp
print("Pillow", PIL.__version__, "- OK")

## Cell 9 — export `vietocr.onnx` + `vietocr_vocab.txt`

Bốn cái bẫy dưới đây tôi đã gặp thật; script bên dưới xử lý cả bốn.

**1. Causal mask là BẮT BUỘC, không được bỏ.** Tôi từng lập luận rằng bỏ được vì
C++ chỉ đọc bước cuối, mà ở vị trí cuối query attend tới toàn bộ key `0..L-1` —
đúng bằng những gì mask cho phép. **Lập luận đó chỉ đúng với decoder MỘT lớp.**
`vgg_transformer` có 6 lớp: đầu ra lớp 2 ở vị trí cuối phụ thuộc đầu ra lớp 1 ở
*mọi* vị trí, mà các vị trí đó khi không mask đã "nhìn trộm" vị trí sau chúng —
thứ model chưa từng thấy lúc huấn luyện. Đo trực tiếp: logits bước cuối lệch
`2.8e-01` so với `forward_decoder` của vietocr, còn có mask thì lệch `0.0`.
Triệu chứng khi chạy thật: ký tự đầu đúng rồi càng về sau càng loạn, kết thúc
bằng những đoạn lặp vô nghĩa.

**2. `dynamo=True`, không phải exporter tracing.** Tracing **đóng băng độ dài
chuỗi** trong `nn.MultiheadAttention` — cả phía decoder (`tgt`) lẫn phía encoder
(suy từ bề rộng ảnh). Model vẫn nạp được, dynamic axes vẫn hiện đúng, nhưng chạy
với kích thước khác lúc trace là chết:
`Reshape ... Input shape:{150,1,256}, requested shape:{80,8,32}`. Nâng opset lên
17/18 không cứu được.

**3. `tgt_is_causal=True` khi gọi decoder.** Thiếu nó, `_detect_is_causal_mask()`
của torch **so sánh nội dung** mask rồi gọi `.item()`, sinh symbol chưa xác định
và `torch.export` bỏ cuộc:
`Could not guard on data-dependent expression Eq(u0, 1)`.

**4. Không gọi thẳng `model.cnn(img)`.** `Vgg.forward` kết thúc bằng
`permute(-1, 0, 1)` — chỉ số **âm**. ONNX bắt buộc `perm` không âm nên model
export ra không nạp nổi: `Invalid attribute perm {-1, 0, 1}`.

Cuối cùng, exporter `dynamo` tách trọng số ra file `.data` riêng; cell gộp lại
thành **một file `.onnx` duy nhất**.

**Hợp đồng tensor** mà `OnnxVietOCR.cpp` giả định:

| Tensor | Kiểu | Ghi chú |
| --- | --- | --- |
| `img` | `float32 [1, 3, 32, W]` | RGB, **chỉ chia 255** → `[0, 1]` |
| `tgt` | `int64 [1, L]` | chuỗi id đã sinh, **batch-major** |
| `logits` | `float32 [1, L, V]` | C++ chỉ đọc `logits[0, -1]` |

In [ ]:
import math, pathlib, torch, onnx
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

MODELS = pathlib.Path("/content/OCR_Extractor/cpp/models")

config = Cfg.load_config_from_name("vgg_transformer")
config["device"] = "cpu"
config["cnn"]["pretrained"] = False
config["predictor"]["beamsearch"] = False

predictor = Predictor(config)          # tải weights vgg_transformer.pth (~151 MB)
model, vocab = predictor.model, predictor.vocab
model.eval()

# --- Bảng ký tự: thứ tự PHẢI khớp id của vietocr.model.vocab.Vocab -------------
#     i2c[0..3] = <pad> <sos> <eos> * ; từ id 4 trở đi là config["vocab"]
lines = ["<pad>", "<sos>", "<eos>", "*"] + list(vocab.chars)
assert len(lines) == len(vocab), (len(lines), len(vocab))
assert all(vocab.i2c[i] == lines[i] for i in range(4, len(lines)))
(MODELS / "vietocr_vocab.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("vocab:", len(lines), "dòng")


class VietOcrOnnx(torch.nn.Module):
    """img [1,3,32,W] float32 [0,1] + tgt int64 [1,L]  ->  logits [1,L,V]."""

    def __init__(self, base):
        super().__init__()
        self.cnn = base.cnn
        self.tf = base.transformer

    def forward(self, img, tgt):
        # Không gọi self.cnn(img): Vgg.forward dùng permute(-1, 0, 1) với chỉ số
        # ÂM, mà ONNX bắt buộc perm không âm -> model export ra không nạp được.
        conv = self.cnn.model.features(img)
        conv = self.cnn.model.dropout(conv)
        conv = self.cnn.model.last_conv_1x1(conv)
        src = conv.transpose(2, 3).flatten(2).permute(2, 0, 1)   # (W', 1, C)
        memory = self.tf.forward_encoder(src)
        tgt_tm = tgt.transpose(0, 1)                              # (L, 1) time-major
        emb = self.tf.pos_enc(self.tf.embed_tgt(tgt_tm) * math.sqrt(self.tf.d_model))
        # CAUSAL MASK LÀ BẮT BUỘC - xem giải thích ở trên.
        # Dựng từ chính emb (new_zeros) chứ không dùng torch.full((L, L)): cách
        # sau tạo kích thước ký hiệu mới mà torch.export không nuốt được.
        n = emb.size(0)
        mask = torch.triu(emb.new_zeros(n, n) + float("-inf"), diagonal=1)
        # tgt_is_causal=True để bỏ qua _detect_is_causal_mask() - hàm đó so sánh
        # NỘI DUNG mask rồi gọi .item(), sinh symbol chưa xác định làm hỏng export.
        out = self.tf.transformer.decoder(emb, memory, tgt_mask=mask, tgt_is_causal=True)
        return self.tf.fc(out.transpose(0, 1))                    # (1, L, V)


wrapper = VietOcrOnnx(model).eval()
onnx_path = MODELS / "vietocr.onnx"

with torch.no_grad():
    torch.onnx.export(
        wrapper,
        (torch.rand(1, 3, 32, 160), torch.tensor([[1, 5, 9]], dtype=torch.int64)),
        str(onnx_path),
        input_names=["img", "tgt"],
        output_names=["logits"],
        dynamic_axes={"img": {3: "width"}, "tgt": {1: "length"}, "logits": {1: "length"}},
        opset_version=18,
        dynamo=True,          # BẮT BUỘC: exporter tracing đóng băng độ dài chuỗi
    )

# dynamo tách trọng số ra <tên>.onnx.data; gộp lại thành một file duy nhất.
external = pathlib.Path(str(onnx_path) + ".data")
if external.exists():
    onnx.save(onnx.load(str(onnx_path)), str(onnx_path), save_as_external_data=False)
    external.unlink()

print("Đã ghi:", onnx_path)
!ls -lh /content/OCR_Extractor/cpp/models/

## Cell 10 — kiểm chứng export (**đừng bỏ qua cell này**)

Ba phép kiểm tra, từ chặt tới lỏng:

1. **Tiền xử lý** của C++ so với `process_input()` của vietocr — phải khớp gần như
   tuyệt đối.
2. **Logits** của ONNX so với chính wrapper torch, trên nhiều cặp `(bề rộng ảnh,
   độ dài chuỗi)` khác nhau. Đây là phép chặt nhất và là phép bắt được lỗi đóng
   băng độ dài ở mục 1 của Cell 9 — nhớ có cả trường hợp `L = 1`.
3. **Chuỗi giải mã** bằng đúng vòng greedy của C++, chạy trên torch và trên ONNX.

Lưu ý: **không** so với `predictor.predict()` của vietocr, vì vòng giải mã của
vietocr chỉ dừng ở `<eos>` còn C++ dừng ở cả `<pad>` lẫn `<eos>`. Khác biệt đó là
về luật dừng, không phải về export, nên đem so sẽ báo động giả.

In [ ]:
import glob, math, pathlib
import numpy as np, onnxruntime as ort, torch
from PIL import Image, ImageDraw, ImageFont
from vietocr.tool.translate import process_input

MODELS = "/content/OCR_Extractor/cpp/models"

# splitlines() chứ KHÔNG phải split("\n"): file kết thúc bằng newline nên
# split("\n") sinh thêm một phần tử rỗng ở cuối, làm lệch số ký tự đi 1.
# splitlines() khớp đúng với std::getline mà loadVocabulary() của C++ dùng.
vocab_lines = open(f"{MODELS}/vietocr_vocab.txt", encoding="utf-8").read().splitlines()
session = ort.InferenceSession(f"{MODELS}/vietocr.onnx", providers=["CPUExecutionProvider"])
vocab_size = session.get_outputs()[0].shape[-1]
print("[0] Input:", {i.name: i.shape for i in session.get_inputs()}, "| vocab:", vocab_size)
assert len(vocab_lines) == vocab_size, (
    f"Bảng ký tự có {len(vocab_lines)} dòng nhưng model xuất {vocab_size} lớp - "
    "export lại Cell 9 hoặc kiểm tra vietocr_vocab.txt")


def preprocess(pil):
    """Bản sao chính xác của prepareCrop() trong OnnxVietOCR.cpp."""
    w, h = pil.size
    new_w = int(32 * w / h)
    new_w = math.ceil(new_w / 10) * 10
    new_w = max(32, min(512, new_w))
    img = pil.convert("RGB").resize((new_w, 32), Image.LANCZOS)
    return (np.asarray(img).transpose(2, 0, 1) / 255.0)[None].astype(np.float32)


# --- 1. Tiền xử lý C++ so với vietocr ----------------------------------------
def load_font(size=26):
    """Font TTF bất kỳ đang có. Ảnh thử trông thế nào không quan trọng - ta so
    HAI HIỆN THỰC trên cùng một ảnh, chứ không chấm điểm độ chính xác OCR."""
    candidates = []
    try:
        import matplotlib
        candidates.append(str(pathlib.Path(matplotlib.get_data_path()) / "fonts/ttf/DejaVuSans.ttf"))
    except Exception:
        pass
    candidates += sorted(glob.glob("/usr/share/fonts/**/DejaVuSans.ttf", recursive=True))
    candidates += sorted(glob.glob("/usr/share/fonts/**/*.ttf", recursive=True))
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()


strip = Image.new("RGB", (420, 48), "white")
ImageDraw.Draw(strip).text((8, 8), "Cước TB 150,534,213", fill="black", font=load_font())

mine, theirs = preprocess(strip), process_input(strip, 32, 32, 512).numpy()
assert mine.shape == theirs.shape, (mine.shape, theirs.shape)
assert np.allclose(mine, theirs, atol=2e-2), np.abs(mine - theirs).max()
print(f"[1] Tiền xử lý khớp vietocr: {mine.shape}, lệch tối đa {np.abs(mine-theirs).max():.2e}")

# --- 2. Wrapper so với CHÍNH forward_decoder của vietocr ---------------------
#     Đây là phép quyết định: nó bắt được ngay nếu wrapper thiếu causal mask
#     (lệch ~2.8e-01) - thứ mà so ONNX với wrapper không bao giờ thấy, vì cả hai
#     cùng sai như nhau.
_tgt = torch.tensor([[1, 5, 7, 4, 9]], dtype=torch.int64)
_img = torch.from_numpy(mine)
with torch.no_grad():
    _mem = model.transformer.forward_encoder(model.cnn(_img))
    _ref = model.transformer.forward_decoder(_tgt.transpose(0, 1), _mem)[0]
    _got = wrapper(_img, _tgt)
_d = (_got - _ref).abs().max().item()
assert _d < 1e-4, f"Wrapper LỆCH {_d:.2e} so với vietocr - thiếu causal mask?"
print(f"[2] Wrapper khớp forward_decoder của vietocr: lệch {_d:.2e}")

# --- 3. Logits ONNX so với torch ---------------------------------------------
for width, length in ((160, 1), (280, 5), (500, 17)):
    img = np.random.rand(1, 3, 32, width).astype(np.float32)
    tgt = np.random.randint(4, vocab_size, size=(1, length)).astype(np.int64)
    tgt[0, 0] = 1
    got = session.run(None, {"img": img, "tgt": tgt})[0]
    with torch.no_grad():
        want = wrapper(torch.from_numpy(img), torch.from_numpy(tgt)).numpy()
    assert got.shape == want.shape == (1, length, vocab_size), (got.shape, want.shape)
    diff = np.abs(got - want).max()
    assert diff < 2e-3, f"LỆCH {diff} ở (width={width}, len={length})"
    print(f"[3] logits khớp torch: width={width:3} len={length:2} lệch {diff:.2e}")

# --- 4. Vòng greedy y hệt C++, chạy trên cả hai ------------------------------
def greedy(step_fn, x, max_len=128):
    ids, text = [1], ""
    for _ in range(max_len):
        logits = step_fn(x, ids)
        nxt = int(logits[0, -1].argmax())
        if nxt in (0, 2):                       # <pad> hoặc <eos>
            break
        text += vocab_lines[nxt]
        ids.append(nxt)
    return text


onnx_text = greedy(lambda x, i: session.run(
    None, {"img": x, "tgt": np.array([i], dtype=np.int64)})[0], mine)
with torch.no_grad():
    torch_text = greedy(lambda x, i: wrapper(
        torch.from_numpy(x), torch.tensor([i], dtype=torch.int64)).numpy(), mine)

print(f"[4] ONNX : {onnx_text!r}")
print(f"    torch: {torch_text!r}")
assert onnx_text == torch_text, "LỆCH - export sai, dừng lại đừng chạy tiếp"
print("\n✅ Export khớp torch. Bản C++ sẽ chạy đúng.")

## Cell 11 — build lại với ONNXRuntime

`--list-engines` phải in ra **cả hai** engine. Nếu chỉ thấy `tesseract` thì CMake
chưa tìm thấy ONNXRuntime hoặc thiếu file trong `models/`.

In [ ]:
%cd /content/OCR_Extractor/cpp
!cmake -B build -DONNXRUNTIME_ROOT_DIR=/opt/onnxruntime
!cmake --build build -j2
!./build/ctkm_extractor --list-engines

## Cell 12 — chạy bằng engine mặc định

`--strict-engine` **cấm fallback**: nếu thấy dòng `Dùng OCR provider 'paddle_vietocr'`
là chắc chắn đang chạy đúng engine, còn thiếu gì nó báo lỗi thay vì âm thầm chuyển
sang Tesseract.

Lần chạy đầu chậm hơn hẳn vì mỗi vùng text detect được đều phải chạy qua vòng giải
mã tự hồi quy của VietOCR.

In [ ]:
!./build/ctkm_extractor --image /content/input.png --out /content/result_vietocr.json \
                        --engine paddle_vietocr --strict-engine --debug

import json, pathlib

out = pathlib.Path("/content/result_vietocr.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


---
# Hồ sơ PDF nhiều trang

Bản C++ **không đọc PDF** (sẽ phải kéo thêm poppler/mupdf làm dependency hệ
thống). Render trước bằng `pdftoppm` rồi truyền từng trang qua `--image` — cờ này
**lặp lại được**.

Bảng CTKM thường chỉ nằm ở **một hoặc vài trang** trong cả tập hồ sơ, nên chương
trình không đoán trang nào chứa bảng mà **chấm điểm từng trang**:

1. Điểm mỗi trang = *(số field trích được, điểm khớp trung bình)*. Trang không có
   bảng ra `(0, 0.0)` nên không bao giờ được chọn khi có trang khác.
2. **Trang chính** = trang điểm cao nhất; hoà thì lấy trang số nhỏ hơn.
3. Field nào **vẫn thiếu** mới lấy bù từ trang khác — cho trường hợp bảng bị tách
   qua nhiều trang. Bước này **không ghi đè** giá trị của trang chính.

Thuật toán này giống hệt bản Python (`mergePageResults`).

In [ ]:
!apt-get -qq install -y poppler-utils      # cung cấp pdftoppm

from google.colab import files

uploaded = files.upload()                  # chọn file PDF nhiều trang
pdf_path = "/content/" + list(uploaded)[0]
print("Đã nhận:", pdf_path)

In [ ]:
import glob, subprocess

# 300 DPI - đúng mức đã kiểm chứng cả pipeline.
subprocess.run(["pdftoppm", "-r", "300", "-png", pdf_path, "/content/trang"],
               check=True)
pages = sorted(glob.glob("/content/trang-*.png"))
print(f"Đã render {len(pages)} trang:")
for page in pages:
    print(" ", page)

In [ ]:
%cd /content/OCR_Extractor/cpp
import shlex

# --image lặp lại một lần cho mỗi trang. shlex.quote để tên file có dấu cách
# không bị shell tách thành nhiều tham số.
args = " ".join(f"--image {shlex.quote(page)}" for page in pages)

!./build/ctkm_extractor {args} \
    --out /content/result_pdf.json --engine paddle_vietocr --strict-engine --debug

import json, pathlib

out = pathlib.Path("/content/result_pdf.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


Trong log `--debug` sẽ thấy điểm từng trang và trang nào được chọn:

```
INFO ctkm.extraction.extractor: Trang 1 (/content/trang-1.png): trích được 0 field
INFO ctkm.extraction.extractor: Trang 2 (/content/trang-2.png): trích được 9 field
INFO ctkm.extraction.extractor: Trang 3 (/content/trang-3.png): trích được 0 field
INFO ctkm.extraction.extractor: Chọn trang 2 làm trang chính trong 3 trang

Trang            : 2 (chính) trong 3 trang đã OCR
```

Field nào lấy bù từ trang khác sẽ có `trang=N` ở cuối dòng trong mục *Nguồn từng
field*, và một cảnh báo tương ứng ở mục *Cảnh báo*.

## Cell 13 — lưu model sang Drive

Export lại mất ~5 phút mỗi phiên. Lưu một lần rồi lần sau chỉ cần copy ngược lại.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ctkm_models
!cp /content/OCR_Extractor/cpp/models/{det.onnx,vietocr.onnx,vietocr_vocab.txt} \
    /content/drive/MyDrive/ctkm_models/
!ls -lh /content/drive/MyDrive/ctkm_models/

# Phiên sau, thay Cell 7-10 bằng đúng dòng này:
# !cp /content/drive/MyDrive/ctkm_models/* /content/OCR_Extractor/cpp/models/

---
## Sự cố hay gặp

| Triệu chứng | Nguyên nhân / cách xử lý |
| --- | --- |
| `--list-engines` chỉ in `tesseract` sau Cell 11 | Thiếu file trong `models/`, hoặc CMake không thấy ONNXRuntime. Chạy `!ls -lh cpp/models` và xem lại log `cmake` có dòng `-- ONNXRuntime: ...` không |
| Cell 10 báo assert ở mục [2] | Export sai. Kiểm tra đã dùng `dynamo=True` chưa — exporter tracing đóng băng độ dài `tgt` và chỉ lộ ra khi `L` khác lúc trace |
| `Invalid attribute perm {-1, 0, 1}` khi nạp model | Wrapper gọi thẳng `model.cnn(img)`. Phải viết lại phần CNN bằng chỉ số dương như trong Cell 9 |
| `Reshape ... requested shape:{3,8,32}` | Đúng lỗi đóng băng độ dài chuỗi. Dùng `dynamo=True`; nâng opset không cứu được |
| Cell 10 báo `AssertionError` ở dòng `got.shape == want.shape` | Bảng ký tự lệch số dòng so với số lớp đầu ra của model. Cell 10 giờ kiểm tra riêng ngay từ đầu và in ra cả hai con số |
| `OSError: cannot open resource` ở Cell 10 | Image Colab không có font ở đường dẫn cứng. `load_font()` đã tự dò matplotlib → `/usr/share/fonts` → `load_default()`; nếu vẫn lỗi thì bạn đang chạy bản notebook cũ |
| `ModuleNotFoundError: onnxscript` | `dynamo=True` cần gói này — có trong Cell 8 |
| `ImportError: cannot import name 'is_directory' from 'PIL._util'` | Pillow đã bị hạ cấp. Cài lại bằng `--no-deps` như Cell 8, rồi **Runtime → Restart session** và chạy lại từ Cell 8 |
| `ModuleNotFoundError: einops` / `gdown` khi import vietocr | Thiếu bước `pip install einops gdown` ở Cell 8 |
| Cell 9 lỗi tải weights | `vocr.vn` chập chờn. Chạy lại; hoặc tải `vgg_transformer.pth` thủ công rồi đặt `config["weights"] = "/đường/dẫn"` |
| Kết quả toàn `null` | Xem `--debug`, mục *OCR raw text* trước. Text thô đã sai thì vấn đề ở ảnh/OCR, không phải tầng trích xuất |
| Chữ tiếng Việt mất dấu ở đường Tesseract | Thiếu `tesseract-ocr-vie`. Kiểm tra bằng `!tesseract --list-langs` |
| `Không đọc được bảng ký tự VietOCR` | Thiếu `models/vietocr_vocab.txt`, hoặc tên khác. C++ tìm `<model>.vocab` rồi mới tới `models/vietocr_vocab.txt` |
| Build hết RAM / bị kill | Giảm song song: `!cmake --build build -j1` |

## Vì sao chạy chậm hơn bản Python

`OnnxVietOCR::recognize()` truyền **cả ảnh** vào mỗi bước giải mã, nên CNN và encoder
chạy lại từ đầu ở từng ký tự. Bản Python thì chạy encoder một lần rồi chỉ lặp decoder.
Đây là đánh đổi có chủ đích: hợp đồng ONNX một-model đơn giản hơn nhiều so với phải
export và ghép hai session encoder/decoder riêng. Muốn nhanh hơn thì tách thành hai
model và sửa vòng lặp trong `OnnxVietOCR.cpp` — các tầng khác không đổi.